# Наивный байесовский классификатор: быстрый вероятностный прогноз

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Наивный байесовский классификатор».

## Что делает метод

Наивный байесовский классификатор оценивает вероятность принадлежности объекта к классу по теореме Байеса в предположении условной независимости признаков при заданном классе. Несмотря на грубость этого допущения, классификатор часто работает удивительно хорошо, поскольку для верного отнесения важен правильный порядок вероятностей, а не их точные значения.

Метод очень быстр, требует мало данных для обучения и применим к табличным признакам и к текстам. В этом блокноте мы решим задачу классификации текстов и разберём, как читать вероятностный вывод. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте врача, который ставит диагноз по набору симптомов. Он рассуждает примерно так: «Температура 39° — это чаще бывает при гриппе. Кашель — тоже чаще при гриппе. Красная сыпь — реже при гриппе, но часто при кори. Складываю все улики и выбираю наиболее правдоподобный диагноз». Именно так работает наивный байесовский классификатор: он смотрит на каждый признак по отдельности, оценивает, насколько он характерен для каждого класса, и объединяет доказательства по теореме Байеса.

Слово «наивный» означает, что метод предполагает независимость признаков между собой при заданном классе (симптомы не связаны друг с другом). Это грубое упрощение, но на практике метод работает удивительно хорошо, особенно для текстов.

**Ключевые термины:**
- **Теорема Байеса** — правило пересчёта вероятности в свете новых данных: P(класс|признаки) ∝ P(признаки|класс) × P(класс).
- **TF-IDF** — способ превратить текст в числа: каждому слову присваивается вес, учитывающий, как часто оно встречается в документе (TF) и насколько редко — во всей коллекции (IDF). Частые во всей коллекции слова (предлоги, союзы) получают низкий вес.
- **Матрица ошибок** — таблица, показывающая, сколько объектов каждого истинного класса модель отнесла к каждому предсказанному классу; позволяет увидеть, какие классы путаются.
- **Сглаживание Лапласа** — добавление небольшой псевдочастоты к каждому слову, чтобы незнакомые слова не обнуляли всю вероятность.

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем подвыборку открытого корпуса `20 newsgroups` — сообщений из тематических групп. Возьмём четыре далёкие темы (космос, медицина, спортивный хоккей, компьютерная графика). Задача — определить тему сообщения по его тексту.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

categories = ['sci.space', 'sci.med',
              'rec.sport.hockey', 'comp.graphics']
topic_ru = ['космос', 'медицина', 'хоккей', 'графика']

train_raw = fetch_20newsgroups(
    subset='train', categories=categories,
    remove=('headers', 'footers', 'quotes'), random_state=42)
test_raw = fetch_20newsgroups(
    subset='test', categories=categories,
    remove=('headers', 'footers', 'quotes'), random_state=42)

print(f'Обучающих сообщений: {len(train_raw.data)}')
print(f'Тестовых сообщений: {len(test_raw.data)}')
print(f'Темы: {topic_ru}')

### Наглядный «ага»-эксперимент: как байесовский классификатор работает на числовых данных

Чтобы понять принцип метода перед переходом к текстам, посмотрим на гауссовский вариант наивного Байеса на синтетических двумерных данных. Он моделирует распределение каждого признака внутри класса как нормальное и на основе этих распределений принимает решение.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.naive_bayes import GaussianNB
from matplotlib.colors import ListedColormap

rng2 = np.random.default_rng(7)
n_each = 120

# Три класса: три «облака» точек
X2d = np.vstack([
    rng2.multivariate_normal([0, 2], [[1, 0.3], [0.3, 1]], n_each),
    rng2.multivariate_normal([3, 0], [[1, -0.2], [-0.2, 1]], n_each),
    rng2.multivariate_normal([-1, -2], [[0.8, 0], [0, 0.8]], n_each),
])
y2d = np.repeat([0, 1, 2], n_each)
labels2d = ['класс A', 'класс B', 'класс C']

gnb = GaussianNB().fit(X2d, y2d)

h = 0.05
xx2, yy2 = np.meshgrid(
    np.arange(X2d[:, 0].min() - 1, X2d[:, 0].max() + 1, h),
    np.arange(X2d[:, 1].min() - 1, X2d[:, 1].max() + 1, h))
Z2 = gnb.predict(np.c_[xx2.ravel(), yy2.ravel()]).reshape(xx2.shape)

cmap_bg2 = ListedColormap(['#dfe9fb', '#d6efec', '#f4e4cf'])

fig, ax = plt.subplots(figsize=(9.5, 6.2))
ax.contourf(xx2, yy2, Z2, cmap=cmap_bg2, alpha=0.7)

for cls, color, label in zip(
        [0, 1, 2],
        [VIZ['series'][0], VIZ['series'][1], VIZ['series'][2]],
        labels2d):
    m = y2d == cls
    ax.scatter(X2d[m, 0], X2d[m, 1],
               color=color, edgecolor='white', s=30, alpha=0.85,
               label=label, zorder=3)

# Нарисуем эллипсы — контуры распределений классов
from matplotlib.patches import Ellipse
for cls, color in zip([0, 1, 2],
                      [VIZ['series'][0], VIZ['series'][1], VIZ['series'][2]]):
    mean = gnb.theta_[cls]
    std = np.sqrt(gnb.var_[cls])
    ellipse = Ellipse(mean, width=4*std[0], height=4*std[1],
                      edgecolor=color, facecolor='none',
                      linewidth=2.0, linestyle='--', zorder=4)
    ax.add_patch(ellipse)

ax.set_xlabel('Признак 1')
ax.set_ylabel('Признак 2')
ax.set_title('Наивный байесовский классификатор: области решения и распределения классов')
ax.legend(loc='upper right')
ax.grid(False)
fig.tight_layout()
plt.show()

**Как читать этот график.** Цвет фона — область, куда модель будет относить новые точки. Пунктирные эллипсы — контуры нормального распределения, которые наивный Байес смоделировал для каждого класса по двум признакам: центр эллипса — среднее, радиус — разброс (два стандартных отклонения). Именно по тому, в какой эллипс «попадает» новая точка с наибольшей вероятностью, классификатор и принимает решение. Точки за пределами «своего» эллипса — это наблюдения, которые труднее всего классифицировать и где возможны ошибки.

## 4. Применение метода

Текст нельзя подать в модель напрямую — нужно преобразовать слова в числа. Делаем это в два шага:
1. **TF-IDF** (`TfidfVectorizer`) — каждое сообщение превращается в вектор весов слов. Слова, редкие в коллекции, но частые в конкретном сообщении, получают высокий вес. Параметр `min_df=3` исключает слова, встречающиеся менее чем в 3 документах — они слишком редки, чтобы быть информативными.
2. **MultinomialNB** — полиномиальный наивный байесовский классификатор, специально предназначенный для счётчиков слов. Параметр `alpha=0.1` — сглаживание Лапласа: небольшое «страховочное» добавление к частотам, чтобы незнакомые слова не обнуляли вероятность целиком.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    TfidfVectorizer(stop_words='english', min_df=3),
    MultinomialNB(alpha=0.1))
model.fit(train_raw.data, train_raw.target)
print('Классификатор обучен на', len(train_raw.data),
      'сообщениях.')

### Качество классификации и матрица ошибок

Следующая ячейка оценивает модель и строит матрицу ошибок — ключевой инструмент понимания того, где именно классификатор ошибается. На диагонали матрицы — верные ответы, вне диагонали — ошибки: строка указывает истинную тему, столбец — предсказанную.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay)

y_pred = model.predict(test_raw.data)
print(f'Доля верных ответов: {accuracy_score(test_raw.target, y_pred):.3f}\n')
print(classification_report(test_raw.target, y_pred,
                            target_names=topic_ru))

fig, ax = plt.subplots(figsize=(7.6, 6.2))
ConfusionMatrixDisplay.from_predictions(
    test_raw.target, y_pred, display_labels=topic_ru,
    cmap='Blues', ax=ax, colorbar=False)
ax.set_xlabel('Предсказанная тема')
ax.set_ylabel('Истинная тема')
ax.set_title('Матрица ошибок')
ax.grid(False)
fig.tight_layout()
plt.show()

**Как читать матрицу ошибок.** Строки — истинные темы сообщений. Столбцы — предсказанные темы. Диагональные ячейки (от верхнего левого к нижнему правому) — это верные ответы: чем темнее синий, тем больше правильных классификаций. Внедиагональные ячейки — ошибки: например, если ячейка на строке «медицина» и столбце «хоккей» закрашена, значит часть медицинских сообщений модель приняла за хоккейные. Концентрация ошибок в конкретной паре тем сигнализирует об их лексическом сходстве.

### Какие слова характерны для темы

Наивный байесовский классификатор хранит оценки вероятности слов в каждом классе. По ним легко извлечь слова, наиболее характерные для темы, — это делает модель прозрачной.

In [ ]:
vec = model.named_steps['tfidfvectorizer']
clf = model.named_steps['multinomialnb']
vocab = np.array(vec.get_feature_names_out())

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.2))
for ax, cls in zip(axes.ravel(), range(len(topic_ru))):
    top = np.argsort(clf.feature_log_prob_[cls])[-10:]
    ax.barh(vocab[top], clf.feature_log_prob_[cls][top],
            color=VIZ['series'][cls % 4])
    ax.set_title(f'Тема: {topic_ru[cls]}')
    ax.set_xlabel('Логарифм вероятности слова')
    ax.grid(True, axis='x')
fig.tight_layout()
plt.show()

**Как читать эти графики (характерные слова темы).** Каждый из четырёх графиков соответствует одной теме. По горизонтали — логарифм вероятности слова в данном классе (чем ближе к нулю, то есть чем правее, тем более вероятно слово для темы). По вертикали — 10 слов с наибольшей вероятностью. Список характерных слов — это своеобразный «профиль» темы глазами модели: именно по этим словам она отличает тему от остальных. Если вы видите нерелевантные слова в списке — это указывает на «шум» в данных или на лексическое пересечение тем.

### Вероятностный вывод для одного сообщения

Классификатор выдаёт не только метку, но и вероятности всех классов. Это позволяет оценить уверенность прогноза.

In [ ]:
sample = ('The telescope captured new images of a distant galaxy '
          'and its orbiting planets.')
proba = model.predict_proba([sample])[0]

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.bar(topic_ru, proba, color=VIZ['series'][0])
ax.set_ylabel('Вероятность темы')
ax.set_title('Распределение вероятностей для примера сообщения')
ax.set_ylim(0, 1)
for i, p in enumerate(proba):
    ax.text(i, p + 0.02, f'{p:.2f}', ha='center')
fig.tight_layout()
plt.show()
print('Предсказанная тема:', topic_ru[int(np.argmax(proba))])

**Как читать график вероятностей для примера.** Столбики показывают, как модель распределяет «уверенность» между темами для одного конкретного сообщения. Высокий столбик у одной темы означает, что классификатор уверен. Примерно равные столбики нескольких тем — сообщение неоднозначное. Обратите внимание: наивный Байес склонен к чрезмерной уверенности (один столбик почти на всю высоту) — это следствие нарушения допущения независимости признаков.

## 5. Интерпретация результата

- **Доля верных ответов и отчёт по классам**: оценивают качество на новых сообщениях. Для далёких тем наивный Байес обычно даёт высокую точность.
- **Матрица ошибок**: на диагонали — верные ответы; скопление ошибок в одной паре тем указывает, что темы лексически близки.
- **Характерные слова**: списки слов с наибольшей вероятностью в классе делают модель содержательно прозрачной — видно, по каким признакам принимается решение.
- **Вероятности классов**: уверенный прогноз даёт вероятность, близкую к единице у одного класса; близкие вероятности нескольких классов сигнализируют о неоднозначном сообщении.

Из-за допущения независимости признаков вероятности наивного Байеса часто чрезмерно уверенны; при необходимости их калибруют.

## 6. Попробуйте на своих данных

Замените демонстрационный корпус своими текстами. Нужны два списка: тексты и их метки классов.

1. Загрузите файл в Colab через вкладку «Файлы».
2. Снимите комментарии в ячейке ниже и укажите имена столбцов с текстом и меткой.
3. Выполните ячейки раздела 4.

## 7. Поэкспериментируйте сами

1. **Измените сглаживание.** В ячейке раздела 4 задайте `alpha=0` (без сглаживания) и `alpha=1.0` (классическое сглаживание Лапласа). Как изменится качество? При `alpha=0` модель рискует «сломаться» на словах, не встречавшихся в обучающей выборке.

2. **Добавьте близкие темы.** Замените одну из четырёх тем на `sci.crypt` (криптография) или `sci.electronics` (электроника). Как изменится матрица ошибок? Близкие темы труднее разделить.

3. **Проверьте своё предложение.** Замените переменную `sample` в ячейке «Вероятностный вывод» на текст из вашей области. Посмотрите, к какой теме модель его отнесёт и насколько она уверена.

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv('my_texts.csv')
# text_column, label_column = 'текст', 'тема'
#
# from sklearn.model_selection import train_test_split
# tr, te = train_test_split(df, test_size=0.3, random_state=42,
#                           stratify=df[label_column])
# class TextBunch: pass
# train_raw, test_raw = TextBunch(), TextBunch()
# train_raw.data = tr[text_column].tolist()
# test_raw.data = te[text_column].tolist()
# Метки переведите в целые коды и задайте список topic_ru.
# Далее повторите ячейки раздела 4.

## 8. Краткие выводы

- Наивный байесовский классификатор применяет теорему Байеса, предполагая условную независимость признаков. Это грубое допущение, но метод часто работает хорошо на практике.
- Для текстов метод особенно хорош: при разбиении текста на слова (признаки) независимость нарушается меньше, чем для структурированных данных.
- «Прозрачность» модели — сильная сторона: характерные слова каждого класса позволяют проверить, правильно ли модель выучила задачу.
- Вероятности классов бывают плохо калиброваны (слишком уверены): если нужны точные вероятности — применяйте калибровку.
- Метод не улавливает взаимодействия между признаками; если они важны, используйте его как быстрый эталон и сравните с более мощными моделями.